# PPS-SOHO Phase A2: locked train-only rank scaling
This is a post-Phase-A diagnostic, not a new hyperparameter search. It fixes lambda=1 and gamma=1, evaluates ranks 64/128/256, physically hides `test.pt`, and never reports CIFAR-100 test accuracy.

In [ ]:
# === Edit this cell only ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/pps-soho'
WORK_DIR = '/content/SOHO-CL'
CACHE_DIR = '/content/tsoho_cifar100_cache'
DRIVE_CACHE_DIR = '/content/drive/MyDrive/T-SOHO/tsoho_cifar100_cache'
OUTPUT_DIR = '/content/pps_soho_phasea2_outputs'
SEED = 1993
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'

In [ ]:
# Fresh clone and environment identity.
import json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
shutil.rmtree(WORK_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_GIT_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
from google.colab import drive
drive.mount('/content/drive')
RUNNER_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('runner commit:', RUNNER_COMMIT)
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# Restore the exact frozen-feature cache from Drive when available; extract only as fallback.
cache = Path(CACHE_DIR); drive_cache = Path(DRIVE_CACHE_DIR)
cache.mkdir(parents=True, exist_ok=True)
if not (cache/'metadata.json').is_file() and (drive_cache/'metadata.json').is_file():
    print('Restoring frozen-feature cache from Drive...', flush=True)
    for name in ('metadata.json', 'train.pt', 'test.pt', 'test.locked.pt'):
        source = drive_cache/name
        if source.is_file():
            print('  copy', name, flush=True); shutil.copy2(source, cache/name)
if not (cache/'metadata.json').is_file():
    print('No reusable cache found; downloading CIFAR-100 and extracting once.', flush=True)
    import kagglehub
    from huggingface_hub import hf_hub_download
    downloaded = Path(kagglehub.dataset_download('zaphat206/cifar-100'))
    candidates = [downloaded, *downloaded.rglob('cifar-100')]
    cifar_dir = next(p for p in candidates if (p/'train').is_file() and (p/'test').is_file() and (p/'meta').is_file())
    checkpoint = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
    command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--root', str(cifar_dir), '--backbone-checkpoint', checkpoint, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', CACHE_DIR, '--output-dir', f'{OUTPUT_DIR}/cache_extract', '--dataset', 'CIFAR-100', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', str(SEED), '--num-classes', '100', '--num-tasks', '10', '--device', 'cuda', '--batch-size', '128', '--num-workers', '2']
    subprocess.run(command, check=True)
metadata = json.loads((cache/'metadata.json').read_text())
assert metadata['dataset'] == 'CIFAR-100'
assert metadata['feature_dim'] == 768 and metadata['finite'] is True
assert metadata['checkpoint_sha256'] == CHECKPOINT_SHA256
print(json.dumps(metadata, indent=2))

In [ ]:
# Focused correctness gate. Sparse-CSC beta warnings are non-fatal.
command = [sys.executable, '-m', 'pytest', '-q', 'tests/test_pps_soho_math.py', 'tests/test_pps_soho_learner.py', 'tests/test_experiment_runner.py']
print('Running:', ' '.join(command), flush=True)
subprocess.run(command, check=True)

In [ ]:
# Lock held-out features and run exactly 8 fixed candidates with task-level progress.
test_path = cache/'test.pt'; locked_test_path = cache/'test.locked.pt'
if test_path.is_file(): test_path.replace(locked_test_path)
assert locked_test_path.is_file() and not test_path.exists(), 'Held-out test must remain physically hidden'
selection_path = Path(OUTPUT_DIR)/'selection.json'
command = [sys.executable, '-u', 'tools/experiment_runner.py', '--select-config', '--config', 'configs/pps_soho_cifar100_rank_scaling.json', '--feature-cache-dir', CACHE_DIR, '--output-dir', str(Path(OUTPUT_DIR)/'selection'), '--selection-output', str(selection_path), '--device', 'cuda']
print('Starting Phase A2: 8 candidates. Each candidate prints START, 10 UPDATE lines, then DONE.', flush=True)
started = time.time(); subprocess.run(command, check=True)
print(f'Rank-scaling diagnostic complete in {(time.time()-started)/60:.1f} minutes')

In [ ]:
# Compact report and locked decision rule. This cell never restores or opens test.pt.
import pandas as pd
payload = json.loads(selection_path.read_text())
table = pd.DataFrame(payload['candidates'])
display(table[['method','rank','ridge_lambda','pps_gamma','validation_average_accuracy','persistent_state_bytes','solver_relative_residual_max','covariance_error_bound','candidate_seconds']])
fly = table[table.method.eq('cached_flycl')].iloc[0]
raw = table[table.method.eq('sft_raw_ridge')].iloc[0]
protected_256 = table[table.method.eq('pps_class_protected') & table['rank'].eq(256)].iloc[0]
standard_256 = table[table.method.eq('pps_standard_fd') & table['rank'].eq(256)].iloc[0]
gap_to_fly = float(fly.validation_average_accuracy - protected_256.validation_average_accuracy)
gap_to_raw = float(raw.validation_average_accuracy - protected_256.validation_average_accuracy)
if gap_to_fly <= 0.50:
    decision = 'REVIEW_FOR_HELDOUT_AUTHORIZATION'
elif gap_to_fly <= 1.00:
    decision = 'INCONCLUSIVE_NO_TEST'
else:
    decision = 'STOP_PPS_AS_PRIMARY_ACCURACY_METHOD'
gates = {
    'numerical_stability': bool(table[table.method.str.startswith('pps_')].solver_relative_residual_max.max() <= 1e-4),
    'rank256_within_0.50pp_of_fly': bool(gap_to_fly <= 0.50),
    'rank256_within_1.00pp_of_fly': bool(gap_to_fly <= 1.00),
    'rank256_beats_standard_fd': bool(protected_256.validation_average_accuracy > standard_256.validation_average_accuracy),
    'state_smaller_than_fly': bool(protected_256.persistent_state_bytes < fly.persistent_state_bytes),
    'heldout_test_remained_locked': bool(locked_test_path.is_file() and not test_path.exists()),
}
gate_payload = {'phase':'PPS-SOHO Phase A2 train-only rank scaling','decision':decision,'gap_to_fly_pp':gap_to_fly,'gap_to_raw_pp':gap_to_raw,'gates':gates,'run_provenance':payload['run_provenance']}
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
(Path(OUTPUT_DIR)/'gate_results.json').write_text(json.dumps(gate_payload, indent=2))
print(json.dumps(gate_payload, indent=2))
print('Do not evaluate test without a separate review and authorization.')

In [ ]:
# Download train-only evidence only.
artifact = shutil.make_archive('/content/pps_soho_phasea2_rank_scaling', 'zip', OUTPUT_DIR)
print('artifact:', artifact)
from google.colab import files
files.download(artifact)